# Approach 1: Singapore Station Observation Cleaning

This notebook is a walkthrough for the first station-observation pipeline in `approach1`.

The goal is not to produce an initial condition yet. The goal is to retrieve and clean sparse Singapore station observations into a structure that a future encoder can consume.

The pipeline is:

```text
data.gov.sg realtime endpoints
-> normalized long table
-> cleaned model-ready station tensor
```

A key design choice: the canonical cleaned data is a long table, not only a matrix like `[value, lat, lon]`. The matrix is useful for neural-network input, but it is too lossy as the main cleaned representation because it drops time, station identity, variable names, units, and missingness.

## 1. Setup

This cell makes the imports work whether the notebook is opened from the repository root or from inside the `approach1` folder.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "approach1":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from IPython.display import HTML

from approach1.station_api import (
    AIR_TEMPERATURE,
    RAINFALL,
    RELATIVE_HUMIDITY,
    WIND_DIRECTION,
    WIND_SPEED,
    normalize_station_payload,
    retrieve_air_temperature,
    retrieve_rainfall,
    retrieve_relative_humidity,
    retrieve_wind_direction,
    retrieve_wind_speed,
    retrieve_all_station_observations,
)
from approach1.station_cleaning import FEATURE_NAMES, station_observations_to_tensor

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)

## 2. Retrieval Functions

There is one function per realtime endpoint:

- `retrieve_air_temperature(date_time)`
- `retrieve_rainfall(date_time)`
- `retrieve_relative_humidity(date_time)`
- `retrieve_wind_speed(date_time)`
- `retrieve_wind_direction(date_time)`

There is also a combined function:

- `retrieve_all_station_observations(date_time)`

Each retrieval function returns a normalized `pandas.DataFrame` with the same schema. That matters because every endpoint uses the same API shape but different physical meaning and units.

## 3. Live API Example

Set `RUN_LIVE_API = True` to call data.gov.sg. By default this is off so the notebook can be opened and run without network dependency.

Use Singapore-local date-times, for example `2026-05-07T16:00:00+08:00`.

In [ ]:
RUN_LIVE_API = True
REQUESTED_TIME = "2026-05-07T16:00:00+08:00"

if RUN_LIVE_API:
    try:
        live_df = retrieve_all_station_observations(REQUESTED_TIME)
        display(live_df.head(130))
        print("rows, columns:", live_df.shape)
        print("variables:", sorted(live_df["variable"].unique()))
    except Exception as exc:
        print("Live API retrieval failed:", repr(exc))
        print("Check internet access, the timestamp format, and whether data.gov.sg is reachable.")
else:
    print("Live API call skipped. Set RUN_LIVE_API = True to retrieve data.gov.sg observations.")


In [ ]:
unique_list = live_df['station_name'].unique()
print(sorted(unique_list))

## 4. Individual Endpoint Runs

Run these cells before the combined retrieval when debugging. If one endpoint fails or returns fewer stations, this makes the problem visible immediately.

Each function returns the same normalized long-table schema, but each variable has different units and physical meaning.

In [ ]:
if RUN_LIVE_API:
    temperature_df = retrieve_air_temperature(REQUESTED_TIME)
    print("temperature rows:", temperature_df.shape)
    display(temperature_df.head(16))
else:
    print("Skipped. Set RUN_LIVE_API = True to retrieve air temperature.")


In [ ]:
if RUN_LIVE_API:
    rainfall_df = retrieve_rainfall(REQUESTED_TIME)
    print("rainfall rows:", rainfall_df.shape)
    display(rainfall_df.head())
else:
    print("Skipped. Set RUN_LIVE_API = True to retrieve rainfall.")


In [ ]:
if RUN_LIVE_API:
    humidity_df = retrieve_relative_humidity(REQUESTED_TIME)
    print("humidity rows:", humidity_df.shape)
    display(humidity_df.head())
else:
    print("Skipped. Set RUN_LIVE_API = True to retrieve humidity.")


In [ ]:
if RUN_LIVE_API:
    wind_speed_df = retrieve_wind_speed(REQUESTED_TIME)
    print("wind speed rows:", wind_speed_df.shape)
    display(wind_speed_df.head())
else:
    print("Skipped. Set RUN_LIVE_API = True to retrieve wind speed.")


In [ ]:
if RUN_LIVE_API:
    wind_direction_df = retrieve_wind_direction(REQUESTED_TIME)
    print("wind direction rows:", wind_direction_df.shape)
    display(wind_direction_df.head())
else:
    print("Skipped. Set RUN_LIVE_API = True to retrieve wind direction.")


## 5. Combined Endpoint Run

After checking individual endpoints, combine them into one long table. This should be the normal input to the cleaning/tensor step.

In [ ]:
if RUN_LIVE_API:
    combined_live_df = retrieve_all_station_observations(REQUESTED_TIME)
    print("combined rows:", combined_live_df.shape)

    html_table = combined_live_df.to_html(
        max_rows=None,
        max_cols=None,
        index=True,
        escape=False,
    )
    display(
        HTML(
            f"""
            <div style="max-height: 520px; overflow: auto; width: 100%;">
                {html_table}
            </div>
            """
        )
    )

    display(combined_live_df.groupby("variable").size().to_frame("rows"))
else:
    print("Skipped. Set RUN_LIVE_API = True to retrieve all endpoints.")


## 6. Offline Example Payloads

The cells below use small API-shaped sample payloads. This lets us inspect the cleaning behavior without depending on the live API.

The examples intentionally include two stations and multiple variables so we can check station ordering, spatial features, time features, missing values, unit conversion, and wind direction encoding.

In [ ]:
def make_payload(values_by_station, *, unit, reading_type):
    stations = [
        {
            "id": "S001",
            "deviceId": "S001",
            "name": "Station One",
            "location": {"latitude": 1.30, "longitude": 103.80},
        },
        {
            "id": "S002",
            "deviceId": "S002",
            "name": "Station Two",
            "location": {"latitude": 1.38, "longitude": 103.90},
        },
    ]
    readings = [
        {
            "timestamp": "2026-05-07T16:00:00+08:00",
            "data": [
                {"stationId": station_id, "value": value}
                for station_id, value in values_by_station.items()
            ],
        }
    ]
    return {
        "code": 0,
        "data": {
            "stations": stations,
            "readings": readings,
            "readingType": reading_type,
            "readingUnit": unit,
        },
        "errorMsg": "",
    }


sample_payloads = {
    AIR_TEMPERATURE: make_payload({"S001": 31.0, "S002": 29.5}, unit="deg C", reading_type="DBT 1M F"),
    RELATIVE_HUMIDITY: make_payload({"S001": 72.0, "S002": 88.0}, unit="percentage", reading_type="RH 1M F"),
    RAINFALL: make_payload({"S001": 0.0, "S002": 2.4}, unit="mm", reading_type="TB1 Rainfall 5 Minute Total F"),
    WIND_SPEED: make_payload({"S001": 10.0}, unit="knots", reading_type="Wind Speed AVG(S)10M M1M"),
    WIND_DIRECTION: make_payload({"S001": 90.0, "S002": 350.0}, unit="degrees", reading_type="Wind Dir AVG (S) 10M M1M"),
}

## 7. Normalize Into a Long Table

The long table is the clean source of truth. Each row is one station-variable observation at one observed time.

This format is easier to audit than a raw matrix because every value keeps its station, time, unit, and physical variable.

In [ ]:
tables = [
    normalize_station_payload(payload, variable, REQUESTED_TIME)
    for variable, payload in sample_payloads.items()
]
clean_long_df = pd.concat(tables, ignore_index=True)
clean_long_df

The expected columns are:

- `requested_time`: the time we asked the API for
- `observed_time`: the timestamp returned by the API
- `station_id`, `station_name`, `latitude`, `longitude`: station metadata
- `variable`: normalized variable name used by our code
- `value`: raw numeric observation from the endpoint
- `unit`, `reading_type`: physical meaning from the API response

In [ ]:
clean_long_df.dtypes

## 8. Convert to Model Input

`station_observations_to_tensor` converts the long table into arrays suitable for a neural-network encoder.

It returns:

- `features`: `[n_stations, n_features]`
- `mask`: same shape, `1` when the feature is present and valid, `0` when missing or invalid
- `coords`: `[n_stations, 2]`, storing raw `[latitude, longitude]` for inspection and mapping
- `station_ids`: deterministic row order metadata, not a model feature
- `feature_names`: deterministic feature order

This is better than a plain `[value, lat, lon]` matrix because missing values are explicit, different physical variables are separate channels, and location/time context is encoded numerically.

## 8A. Final Feature Design

Current decision: use raw `latitude` and `longitude` directly as model features. Do not use station name, station id, unit, or reading type as model features. Those stay as metadata for debugging and validation.

The model-facing input has three ideas:

1. `features`: numeric values the encoder can use.
2. `mask`: whether each feature value exists and passed cleaning.
3. metadata: station ids, names, units, and reading types for inspection, not training features.

For one timestamp, the clean model tensor can also be viewed as `[n_stations, n_features, 2]`, where the last dimension is `[value, mask]`.

In [ ]:
MODEL_FEATURES_WITH_PROVENANCE = [
    "latitude",                  # raw station latitude from data.gov.sg station metadata
    "longitude",                 # raw station longitude from data.gov.sg station metadata
    "hour_sin",                  # sin(2*pi*observed_hour/24), from observed_time
    "hour_cos",                  # cos(2*pi*observed_hour/24), from observed_time
    "dayofyear_sin",             # sin(2*pi*day_of_year/366), from observed_time
    "dayofyear_cos",             # cos(2*pi*day_of_year/366), from observed_time
    "temperature_c",             # air_temperature endpoint value, unit deg C
    "relative_humidity_pct",     # relative-humidity endpoint value, unit percentage
    "rainfall_5min_mm",          # rainfall endpoint value, 5-minute accumulation in mm
    "rainfall_present",          # 1 if rainfall_5min_mm > 0 else 0
    "log1p_rainfall_5min_mm",    # log(1 + rainfall_5min_mm)
    "wind_speed_mps",            # wind-speed endpoint value in knots * 0.514444
    "wind_dir_sin",              # sin(wind_direction_degrees), from wind-direction endpoint
    "wind_dir_cos",              # cos(wind_direction_degrees), from wind-direction endpoint
    "wind_u_mps",                # wind_speed_mps * wind_dir_sin
    "wind_v_mps",                # wind_speed_mps * wind_dir_cos
]

MODEL_FEATURES_WITH_PROVENANCE


In [ ]:
feature_catalog = pd.DataFrame(
    [
        ("latitude", "spatial", "Raw station latitude in degrees."),
        ("longitude", "spatial", "Raw station longitude in degrees."),
        ("hour_sin", "time", "Cyclical hour-of-day encoding."),
        ("hour_cos", "time", "Cyclical hour-of-day encoding."),
        ("dayofyear_sin", "time", "Cyclical day-of-year encoding."),
        ("dayofyear_cos", "time", "Cyclical day-of-year encoding."),
        ("temperature_c", "weather", "Surface air temperature in Celsius."),
        ("relative_humidity_pct", "weather", "Surface relative humidity in percent."),
        ("rainfall_5min_mm", "weather", "5-minute rainfall accumulation in mm."),
        ("rainfall_present", "derived", "1 if rainfall_5min_mm > 0 else 0."),
        ("log1p_rainfall_5min_mm", "derived", "log(1 + rainfall_5min_mm)."),
        ("wind_speed_mps", "weather", "Wind speed converted from knots to m/s."),
        ("wind_dir_sin", "weather", "Sine of wind direction degrees."),
        ("wind_dir_cos", "weather", "Cosine of wind direction degrees."),
        ("wind_u_mps", "derived", "wind_speed_mps * wind_dir_sin."),
        ("wind_v_mps", "derived", "wind_speed_mps * wind_dir_cos."),
    ],
    columns=["feature", "group", "meaning"],
)

feature_catalog


In [ ]:
station_tensor = station_observations_to_tensor(clean_long_df)

print("features shape:", station_tensor.features.shape)
print("mask shape:", station_tensor.mask.shape)
print("coords shape:", station_tensor.coords.shape)
print("station_ids:", station_tensor.station_ids)
print("feature_names:", station_tensor.feature_names)

In [ ]:
features_df = pd.DataFrame(
    station_tensor.features,
    index=station_tensor.station_ids,
    columns=station_tensor.feature_names,
)
features_df

In [ ]:
mask_df = pd.DataFrame(
    station_tensor.mask,
    index=station_tensor.station_ids,
    columns=station_tensor.feature_names,
)
mask_df

The model can consume `features` and `mask` separately, or we can stack them into one tensor. In the stacked version, the last dimension is `[value, mask]`.

In [ ]:
value_mask_tensor = np.stack(
    [station_tensor.features, station_tensor.mask],
    axis=-1,
)
print("value_mask_tensor shape:", value_mask_tensor.shape)
print("last dimension: [value, mask]")

first_station = station_tensor.station_ids[0]
pd.DataFrame(
    value_mask_tensor[0],
    index=station_tensor.feature_names,
    columns=["value", "mask"],
).rename_axis(f"first station: {first_station}")


In [ ]:
coords_df = pd.DataFrame(
    station_tensor.coords,
    index=station_tensor.station_ids,
    columns=["latitude", "longitude"],
)
coords_df

## 9. Important Cleaning Details

### Station identity, units, and reading type

Station id and station name are not model features. They stay as metadata for row alignment and debugging. Units and `reading_type` also stay as metadata: they tell us how to validate and convert values, but they should not be fed directly into the encoder as strings.

### Spatial features

Raw latitude and longitude are available in `coords`. The encoder-facing `features` matrix uses normalized spatial features, `latitude` and `longitude`, so location is included without putting large raw longitude values next to weather values.

### Time features

Weather has strong time structure. The model input includes cyclical time features: `hour_sin`, `hour_cos`, `dayofyear_sin`, and `dayofyear_cos`. This avoids treating hour `23` and hour `0` as far apart.

### Rainfall derived features

Rainfall is a 5-minute accumulation. The model keeps the raw `rainfall_5min_mm`, adds `rainfall_present`, and adds `log1p_rainfall_5min_mm` so heavy rainfall is less numerically extreme.

### Wind speed

The API returns wind speed in knots. The model feature uses m/s:

```text
wind_speed_mps = wind_speed_knots * 0.514444
```

### Wind direction

Wind direction is circular. Raw degrees are a bad neural-network feature because `359` and `1` are physically close but numerically far apart.

So we encode direction as:

```text
wind_dir_sin = sin(direction_degrees)
wind_dir_cos = cos(direction_degrees)
```

When both speed and direction exist, we also derive standard meteorological wind components:

```text
wind_u_mps = wind_speed_mps * wind_dir_sin
wind_v_mps = wind_speed_mps * wind_dir_cos
```

### Missing values

Missing values are not imputed in this first version. The feature value is set to `0.0`, and the mask tells the model whether the feature is real.

This is important: `0.0` alone is not trustworthy as a missing marker because rainfall can truly be `0.0`.

## 10. Invalid Value Example

The cleaner rejects physically invalid or suspicious values:

- temperature outside `10..45 C`
- humidity outside `0..100 %`
- rainfall below `0`
- wind speed below `0`
- wind direction outside `0..360`

Invalid values become missing: feature value `0.0`, mask value `0`.

In [ ]:
bad_temperature = normalize_station_payload(
    make_payload({"S001": 50.0}, unit="deg C", reading_type="DBT 1M F"),
    AIR_TEMPERATURE,
    REQUESTED_TIME,
)
bad_tensor = station_observations_to_tensor(bad_temperature)

pd.DataFrame(
    bad_tensor.mask,
    index=bad_tensor.station_ids,
    columns=bad_tensor.feature_names,
)

## 11. PyTorch Tensor Option

By default, the output uses NumPy arrays. If training code wants PyTorch tensors directly, pass `as_torch=True`.

In [ ]:
torch_tensor = station_observations_to_tensor(clean_long_df, as_torch=True)
type(torch_tensor.features), torch_tensor.features.shape

## 12. What To Check Before Training

Before using this as encoder input, inspect these things:

1. Are station counts stable enough across timestamps?
2. How often is each feature missing?
3. Are rainfall and wind variables aligned with temperature/humidity timestamps?
4. Should we keep stations with too many missing values?
5. Should coordinates be normalized before the encoder sees them?

A blunt warning: cleaned station observations are not an initial condition. They are sparse surface observations. The encoder can learn a mapping from these observations to an IC only if we later provide paired ground-truth IC targets.